# Focus 
During extreme weather conditions, are different communal zip codes such as ones located in rural, suburban, or urban communities more accident prone 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Load in the datatset 
df = pd.read_csv("../../US_Accidents_March23_sampled_500k.csv")

In [2]:
# Trim zipcodes 
df["Zipcode"] = df["Zipcode"].astype(str).str[:5]

# Check 
print(df["Zipcode"].str.len().value_counts()) 
print(df["Zipcode"].value_counts())

Zipcode
5.0    499884
Name: count, dtype: int64
Zipcode
27610    846
32819    788
33186    745
91761    739
70808    716
        ... 
31309      1
32346      1
88318      1
05743      1
84408      1
Name: count, Length: 17002, dtype: int64


##### Distinguishing commuities (what classifys a zipcode to be urban, suburban or rural)

In [ ]:
# Road features 
road_features = ["Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal", "Turning_Loop"]

# Each feature in a zipcode 
zip_data = df.groupby("Zipcode")[road_features].mean()

# Nummber of accidents in a zipcode 
zip_data["accident_count"] = df.groupby("Zipcode").size()

# Removing zipcodes with less than 10 accidents 17002 --> 7561
zip_data = zip_data[zip_data["accident_count"] >= 10]

#Average of city-type features 
zip_data["urban_score"] = zip_data[["Amenity", "Crossing", "Station", "Stop", "Traffic_Signal"]].mean(axis=1)

print(zip_data["urban_score"].describe())

##### Grouping: Using 25% percentile as low cutoff and 75% as high cutoff 

In [ ]:
# Grouping the different communities 
low_cutoff = 0.0083
high_cutoff = 0.0820

zip_data["Community"] = "Suburban" # middle ground 
zip_data.loc[zip_data["urban_score"] < low_cutoff, "Community"] = "Rural"
zip_data.loc[zip_data["urban_score"] >= high_cutoff, "Community"] = "Urban"

print(zip_data["Community"].value_counts())

##### Grouping weather conditions to make it easier to define whats considered extreme weather 


In [ ]:
# Weather conditions 
cond = pd.read_csv("../../hi.csv") # using Jasper's file
name = cond["Weather_Condition"].str.lower()

# Cut off for wind speed and visibility to clean impossible values 
df.loc[df["Wind_Speed(mph)"] > 100, "Wind_Speed(mph)"] = np.nan # 822 --> 82
df.loc[df["Visibility(mi)"] > 10, "Visibility(mi)"] = np.nan # 130 --> 10

# Intensity 
cond["is_light"] = name.str.contains("light|vicinity") # grouping variables without qualifiers 
cond["is_heavy"] = name.str.contains("heavy") | (
    name.str.contains("thunder|t-storm|tornado|funnel|squall|hail") # including those without qualifiers
    & ~name.str.contains("light|vicinity") # to avoid mislabeling rows (eg. Light Rain with Thunder) 
)

cond["is_moderate"] = (
    name.str.contains("rain|drizzle|shower|snow|sleet|wintry|ice pellets")
    & ~cond["is_light"] & ~cond["is_heavy"] # middle ground between light and heavy
)

# Rain / Snow 
cond["is_rain"] = name.str.contains("rain|showers|drizzle") # rain
cond["is_snow"] = name.str.contains("snow|wintry|sleet|ice pellets|hail|grains") # snow 
cond["is_freezing"] = name.str.contains("freezing|ice pellets|sleet|wintry") # cold 

# Storms 
cond["is_storm"]  = name.str.contains("thunder|t-storm|squall|tornado|funnel")

# Visibilty
cond["is_low_vis"] = name.str.contains("fog|haze|mist|smoke|dust|sand|volcanic|ash")

# Wind 
cond["is_windy"] = name.str.contains("windy|squall|tornado")

# Cloud Conditions 
cond["is_clouds"] = name.str.contains("fair|cloud|clear|overcast|scattered")

# Percepition (has NaN values)
cond["is_unknown"] = name.str.contains("n/a")

cond.sample(10)     

##### Scoring: intensity + add_hazards = serverity score  

In [ ]:
# Scoring intensity 1-4 (precipitation)

cond["intensity"] = np.select(
    [cond["is_heavy"], cond["is_moderate"], cond["is_light"]],
    [3, 2, 1], 
    default=0 # covers non qualifiers (clear,cloudy,fair etc.) 
)

# Conditions that make driving more dangerous (eg. 0=none, 1=freezing, 2=storm,windy)
cond["add_hazards"] = (
    cond["is_freezing"].astype(int) + cond["is_storm"].astype(int)
    + cond["is_low_vis"].astype(int) + cond["is_windy"].astype(int)
)

# severity_score = raw total 
cond["severity_score"] = (cond["intensity"] + cond["add_hazards"]).astype(float)
cond.loc[cond["is_unknown"], "severity_score"] = np.nan    # unknown is not "mild"

s = cond["severity_score"]
cond["weather_level"] = np.select( #weather_level = total sorted into four categories
    [s >= 3, s == 2, s == 1, s == 0],
    [4, 3, 2, 1],         
    default=np.nan          
)

cond[["Weather_Condition", "intensity", "add_hazards", "severity_score", "weather_level"]].sample(20)

In [ ]:
# Attaching weather_level to each accident

lookup = cond[["Weather_Condition", "severity_score", "weather_level"]]
df = df.merge(lookup, on="Weather_Condition", how="left")
print(len(df)) # dataset is still 500000

# Defining extreme weather = level 4

df["is_extreme"] = df["weather_level"] == 4
print(df["weather_level"].value_counts(dropna=False).sort_index())
print("Share extreme:", round(df["is_extreme"].mean(), 4))

# Adding Communities

df["zip5"] = df["Zipcode"].astype(str).str[:5]
df["Community"] = df["zip5"].map(zip_data["Community"])
print("No community:", df["Community"].isna().sum())

# Comparing Communities

community_table = df.groupby("Community")["is_extreme"].agg(["count", "sum", "mean"])
community_table.columns = ["accidents", "extreme_accidents", "share_extreme"]
print(community_table.round(4))

In [ ]:
# bar chart showing community and amount of accidents 
